# RNN 
- 2 Entradas PwmD, PwmE
- 1 Saida $\Delta Y
 $ 
- 1 Camada
- n neurônios

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

Datasets = []
PREDICTORS = ["PwmD", "PwmE"]   
TARGET = ["Y"]       
TIME_STEPS = 7

for i in range(4):
    Dataset = pd.read_excel(f"../../../RotedData/Data.xlsx", f"D{i+1}")   
    # Criar as colunas de deltas automaticamente
    for var in TARGET:
        Dataset[f"Delta{var}"] = Dataset[var].shift(-1) - Dataset[var]    
           
    Datasets.append(Dataset)

for i in range(2):   
    Dataset = pd.read_csv(f"../../../Data/Data{i + 1}.csv")
        
    # Criar as colunas de deltas automaticamente
    for var in TARGET:
        Dataset[f"Delta{var}"] = Dataset[var].shift(-1) - Dataset[var]    
    
    Datasets.append(Dataset)
    

for i in range(len(Datasets)):
    print(f"++++++++++++++++++++ Dataset {i+1} +++++++++++++++++++++++")
    print(Dataset.head(5))

++++++++++++++++++++ Dataset 1 +++++++++++++++++++++++
     X    Y  Theta    Wd   We  WdRef  WeRef   PwmD   PwmE  DeltaY
0  0.0  0.7    0.0  0.00  0.0  -0.00   0.00  -0.00   0.00     0.0
1  0.0  0.7    0.0  0.00  0.0   3.02   3.02  -0.00   0.00     0.0
2  0.0  0.7    0.0  0.00  0.0   3.02   3.02  45.32  45.32     0.0
3  0.0  0.7    0.0  0.03  0.0   3.02   3.02  45.32  45.32     0.0
4  0.0  0.7    0.0  0.00  0.0   3.02   3.02  63.00  63.44     0.0
++++++++++++++++++++ Dataset 2 +++++++++++++++++++++++
     X    Y  Theta    Wd   We  WdRef  WeRef   PwmD   PwmE  DeltaY
0  0.0  0.7    0.0  0.00  0.0  -0.00   0.00  -0.00   0.00     0.0
1  0.0  0.7    0.0  0.00  0.0   3.02   3.02  -0.00   0.00     0.0
2  0.0  0.7    0.0  0.00  0.0   3.02   3.02  45.32  45.32     0.0
3  0.0  0.7    0.0  0.03  0.0   3.02   3.02  45.32  45.32     0.0
4  0.0  0.7    0.0  0.00  0.0   3.02   3.02  63.00  63.44     0.0
++++++++++++++++++++ Dataset 3 +++++++++++++++++++++++
     X    Y  Theta    Wd   We  WdRef  WeRef

In [2]:
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    # Criar as colunas de deltas automaticamente
    for var in TARGET:
        Dataset[f"Delta{var}"] = Dataset[var].shift(-1) - Dataset[var]

    # Dropar apenas uma vez (mantém apenas linhas onde todos os deltas existem)
    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET])

    Datasets[i] = Dataset

    print(f"++++++++++++++++++++ Dataset {i+1} +++++++++++++++++++++++")
    print(Datasets[i].tail(5))


++++++++++++++++++++ Dataset 1 +++++++++++++++++++++++
     Unnamed: 0     X     Y  Theta   Wd   We  WdRef  WeRef   PwmD   PwmE  \
971       67.97  0.01  0.69   -0.8  0.0  0.0   0.85   2.03 -52.65 -29.75   
972       68.04  0.01  0.69   -0.8  0.0  0.0   0.85   2.03 -47.56 -17.83   
973       68.11  0.01  0.69   -0.8  0.0  0.0   0.85   2.03 -47.56 -17.83   
974       68.18  0.01  0.69   -0.8  0.0  0.0   0.85   2.03 -42.47  -5.67   
975       68.25  0.01  0.69   -0.8  0.0  0.0   0.85   2.03 -42.47  -5.67   

     DeltaY  
971     0.0  
972     0.0  
973     0.0  
974     0.0  
975     0.0  
++++++++++++++++++++ Dataset 2 +++++++++++++++++++++++
     Unnamed: 0     X     Y  Theta    Wd    We  WdRef  WeRef    PwmD    PwmE  \
971       67.97  0.04  0.66  -0.78 -2.40 -1.81  -3.20  -3.17 -112.80 -122.28   
972       68.04  0.04  0.67  -0.79 -2.93 -2.43  -3.27  -3.29 -112.80 -122.28   
973       68.11  0.03  0.67  -0.79 -3.29 -3.09  -3.14  -3.17 -107.79 -120.76   
974       68.18  0.03  0.68  

In [3]:
NormDatasets = []
TARGET = ["DeltaY"]

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

# ===== TRAIN FASE 1 =====
Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

# ===== TRAIN FASE 2 =====
Train2 = Datasets[1].copy()
Train2[PREDICTORS] = SCALER.transform(Train2[PREDICTORS])
Train2[TARGET] = OUT_SCALER.transform(Train2[TARGET])
NormDatasets.append(Train2)


for i in range(4):
      CurrentTestDataset = Datasets[i + 2]
      CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
      CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
      NormDatasets.append(CurrentTestDataset)
      print(f"++++++++++++++++++++ Dataset Normalizado {i+2} +++++++++++++++++++++++")
      print(NormDatasets[i].head(5))

++++++++++++++++++++ Dataset Normalizado 2 +++++++++++++++++++++++
   Unnamed: 0    X    Y  Theta    Wd   We  WdRef  WeRef      PwmD      PwmE  \
0        0.00  0.0  0.7   0.00  0.00  0.0   0.00   0.00  0.221564  0.111917   
1        0.07  0.0  0.7   0.00  0.00  0.0   3.28   3.28  0.221564  0.111917   
2        0.14  0.0  0.7   0.00  0.00  0.0   3.28   3.28  0.803114  0.594038   
3        0.21  0.0  0.7   0.00  0.06  0.0   3.28   3.28  0.803114  0.594038   
4        0.28  0.0  0.7   0.01  0.33  0.0   3.28   3.28  1.025343  0.786494   

    DeltaY  
0  0.00264  
1  0.00264  
2  0.00264  
3  0.00264  
4  0.00264  
++++++++++++++++++++ Dataset Normalizado 3 +++++++++++++++++++++++
   Unnamed: 0     X     Y  Theta   Wd   We  WdRef  WeRef      PwmD      PwmE  \
0        0.00  0.01  0.69   -0.8  0.0  0.0   0.85   2.03 -0.219706  0.175547   
1        0.07  0.01  0.69   -0.8  0.0  0.0   0.85   2.03 -0.159602  0.294682   
2        0.14  0.01  0.69   -0.8  0.0  0.0   0.85   2.03 -0.159602  0.294

In [4]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

In [5]:
x_train1, y_train1 = CreateSequences(Train1[PREDICTORS], Train1[TARGET], TIME_STEPS)

x_train2, y_train2 = CreateSequences(Train2[PREDICTORS], Train2[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences((NormDatasets[5])[PREDICTORS], (NormDatasets[5])[TARGET], TIME_STEPS)

print(f"Dimensão da entrada: {np.shape(x_train1)}")
print(f"Dimensão da saida: {np.shape(y_train1)}")

print(f"Dimensão da entrada: {np.shape(x_train2)}")
print(f"Dimensão da saida: {np.shape(y_train2)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(x_val)}")

Dimensão da entrada: (969, 7, 2)
Dimensão da saida: (969, 1)
Dimensão da entrada: (969, 7, 2)
Dimensão da saida: (969, 1)
Dimensão da entrada: (1264, 7, 2)
Dimensão da saida: (1264, 7, 2)


In [6]:
import matplotlib.pyplot as plt

def PlotHistory(history):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.title('Training History')
    plt.legend()
    plt.grid(True)
    plt.show()
    

In [7]:
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np

TITLES = ["train1", "train2", "test1", "test2", "test3", "val"]

def PlotOut(ax, title, target_name, y_true, y_pred):
    time = (np.arange(0, len(y_pred), 1).astype(float) * 0.07).round(5)

    ax.scatter(time, y_true, marker='o', s=12, label='Amostras Reais', alpha=0.7)
    ax.scatter(time, y_pred, marker='x', s=12, label='Valores Preditos', alpha=0.7)
    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)
    ax.legend()
    ax.grid(True)


def EvalModel(model):
    n_datasets = len(Datasets)
    n_targets = len(TARGET)
    fig, axs = plt.subplots(n_datasets, n_targets, figsize=(6 * n_targets, 4 * n_datasets))
    
    metrics = {name: {"R2_train1": [], "R2_train2": [],
                      "R2_test1": [], "R2_test2": [],
                      "R2_test3": [],"R2_val": [],
                      "MSE_train1": [], "MSE_train2": [],
                      "MSE_test1": [], "MSE_test2": [],
                      "MSE_test3": [], "MSE_val": [],} for name in TARGET}

    for i, dataset in enumerate(NormDatasets):
        x = dataset[PREDICTORS]
        y = dataset[TARGET]
        x, y = CreateSequences(x, y, TIME_STEPS)

        y_pred_diff = OUT_SCALER.inverse_transform(model.predict(x))
        y_diff = OUT_SCALER.inverse_transform(y)
        
        y_true = np.zeros_like(y_diff)
        y_pred = np.zeros_like(y_pred_diff)

        # valores iniciais reais (não normalizados)
        init_vals = np.array([
            Datasets[i]["Theta"].iloc[0],
            Datasets[i]["X"].iloc[0],
            Datasets[i]["Y"].iloc[0]
        ])

        # reconstrução cumulativa saída a saída
        for j, name in enumerate(TARGET):
            y_true[:, j] = init_vals[j] + np.cumsum(y_diff[:, j])
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred_diff[:, j])


        # Calcula métricas por saída
        for j, name in enumerate(TARGET):
            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = mean_squared_error(y_true[:, j], y_pred[:, j])
            metrics[name][f"R2_{TITLES[i]}"].append(r2)
            metrics[name][f"MSE_{TITLES[i]}"].append(mse)

            print(f"{name} | {TITLES[i]} -> R² = {r2:.4f}, MSE = {mse:.4e}")
            
            # Seleciona o eixo correto (funciona mesmo com 1x1, 1x2 ou 3x2)
            ax = axs[i][j] if n_datasets > 1 and n_targets > 1 else (
                axs[j] if n_targets > 1 else axs[i] if n_datasets > 1 else axs
            )
            PlotOut(ax, TITLES[i], name, y_true[:, j], y_pred[:, j])

    plt.tight_layout()

    # Retorna métricas médias para análise posterior
    return metrics, plt, fig

In [8]:
import numpy as np
import pandas as pd
from tensorflow import keras
from keras.callbacks import EarlyStopping
from keras import initializers

# tamanho da entrada
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET) 
N_MODELS = 10  # número de inicializações

seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

# arquiteturas das camadas ocultas
neurons = [[5, 2], [10, 5], [15, 5], [20, 5], [20, 10], [25, 10], [10, 5 ,2 ], [15, 10, 5]]

results = {}
excel_file = "resultados.xlsx"

for arch in neurons:  # cada arquitetura
    for i, s in enumerate(seeds):

        initializer = initializers.RandomNormal(seed=int(s))

        model = keras.models.Sequential()
        model.add(keras.layers.Input(shape=(TIME_STEPS, INPUT_SIZE)))

        # adiciona camadas RNN
        for j, n in enumerate(arch):

            # se não for a última camada RNN
            if j < len(arch) - 1:
                model.add(
                    keras.layers.SimpleRNN(
                        n,
                        activation="tanh",
                        return_sequences=True,
                        kernel_initializer=initializer
                    )
                )
            else:
                model.add(
                    keras.layers.SimpleRNN(
                        n,
                        activation="tanh",
                        kernel_initializer=initializer
                    )
                )

        # camada de saída
        model.add(
            keras.layers.Dense(
                OUTPUT_SIZE,
                activation="linear",
                kernel_initializer=initializer
            )
        )
        # salva pesos iniciais
        w0 = model.get_weights()

        # compila
        model.compile(loss="mean_squared_error", optimizer="adam")
        early_stopping_monitor = EarlyStopping(
            monitor='val_loss',
            patience=50,
            restore_best_weights=True
        )

        x_train = np.concatenate((x_train1, x_train2), axis=0)
        y_train = np.concatenate((y_train1, y_train2), axis=0)

        history = model.fit(
            x_train, 
            y_train, 
            epochs=1000,
            callbacks=[early_stopping_monitor],
            validation_data=(x_val, y_val),
            verbose=0
        )
        
    
        # salva pesos finais
        wf = model.get_weights()

       # avalia o modelo
        metrics, plt, fig = EvalModel(model)


        row = {
            "model": f"model_{n}_{i}",
            "Neurons": arch,
            "W0": str([w.round(4).tolist() for w in w0]),
            "Wf": str([w.round(4).tolist() for w in wf]),
        }


        for name in TARGET:
            row.update({
                f"R2_Train_1_{name}": metrics[name]["R2_train1"],
                f"MSE_Train_1_{name}": metrics[name]["MSE_train1"],
                f"R2_Train_2_{name}": metrics[name]["R2_train2"],
                f"MSE_Train_2_{name}": metrics[name]["MSE_train2"],
                f"R2_Val_{name}": metrics[name]["R2_val"],
                f"MSE_Val_{name}": metrics[name]["MSE_val"],
                f"R2_Test_1_{name}": metrics[name]["R2_test1"],
                f"MSE_Test_1_{name}": metrics[name]["MSE_test1"],
                f"R2_Test_2_{name}": metrics[name]["R2_test2"],
                f"MSE_Test_2_{name}": metrics[name]["MSE_test2"],
                f"R2_Test_3_{name}": metrics[name]["R2_test3"],
                f"MSE_Test_3_{name}": metrics[name]["MSE_test3"],
            })
        
        save_fig = any(
            any(v > 0.5 for v in metrics[name][key])
            if isinstance(metrics[name][key], (list, tuple))
            else metrics[name][key] > 0.5
            for name in TARGET
            for key in metrics[name]
            if key.startswith("R2_test"))
    

        if save_fig:
            plt.savefig(
                f"./results/model_{n}_{i}.pdf",
                format="pdf",
                bbox_inches="tight"
            )
            print(f"Figura salva em: ./results/model_{n}_{i}.pdf")

        plt.close(fig)

        df = pd.DataFrame([row])

        # salva/atualiza Excel incrementalmente
        try:
            # tenta abrir existente e adicionar linha
            old = pd.read_excel(excel_file)
            new_df = pd.concat([old, df], ignore_index=True)
            new_df.to_excel(excel_file, index=False)
        except FileNotFoundError:
            # se não existir, cria arquivo novo
            df.to_excel(excel_file, index=False)

        print(f"Modelo {i} treinado e salvo no Excel")





31/31 [==============================] - 0s 2ms/step
DeltaY | train1 -> R² = -1.5472, MSE = 2.1289e-01
31/31 [==============================] - 0s 2ms/step
DeltaY | train2 -> R² = -1.9393, MSE = 3.1471e-01
31/31 [==============================] - 0s 2ms/step
DeltaY | test1 -> R² = -1.7662, MSE = 2.2594e-01
31/31 [==============================] - 0s 2ms/step
DeltaY | test2 -> R² = -1.9264, MSE = 3.0896e-01
45/45 [==============================] - 0s 2ms/step
DeltaY | test3 -> R² = -3.9294, MSE = 3.7729e-01
40/40 [==============================] - 0s 2ms/step
DeltaY | val -> R² = -2.9618, MSE = 2.9403e-01
Modelo 0 treinado e salvo no Excel
31/31 [==============================] - 0s 2ms/step
DeltaY | train1 -> R² = -1.5337, MSE = 2.1176e-01
31/31 [==============================] - 0s 2ms/step
DeltaY | train2 -> R² = -1.9278, MSE = 3.1348e-01
31/31 [==============================] - 0s 2ms/step
DeltaY | test1 -> R² = -1.7530, MSE = 2.2487e-01
31/31 [==============================] - 